![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 04: Data Manipulation)**

**Session 4G: SparkSQL and Data Understanding**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 04.</td>
</tr>
<tr>
<td align="left">Estimated duration</td>
<td>Approximately 40 minutes, based on 240 minutes of M04 practical/self-learning work divided across six M04 notebooks.</td>
</tr>
<tr>
<td align="left">Main packages</td>
<td><code>pyspark</code>, <code>pandas</code>, and Python standard-library path and package handling. Spark also requires a Java runtime.</td>
</tr>
<tr>
<td align="left">Data files</td>
<td><code>mtcars.csv</code> and <code>kddcup.gz</code> from the public <code>Jupyter/data/</code> folder.</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Data Files](#2-setup-and-data-files)
- [3. Spark Session and DataFrame Setup](#3-spark-session-and-dataframe-setup)
- [4. DataFrame Operations on mtcars](#4-dataframe-operations-on-mtcars)
- [5. SQL Queries with Temporary Views](#5-sql-queries-with-temporary-views)
- [6. KDD SparkSQL Application](#6-kdd-sparksql-application)
- [7. Checks and Reflection](#7-checks-and-reflection)
- [8. Reflection and References](#8-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>

### 1. Overview and Learning Goals

This session introduces Spark SQL and Spark DataFrames. You will load a small tabular dataset, create Spark DataFrames, register temporary SQL views, run SQL queries, and use Spark DataFrame operations for exploratory data analysis on a larger network-interaction dataset.

Spark SQL extends Spark with structured data processing. A Spark DataFrame is a distributed table with named columns. It can be queried with SQL or manipulated with a DataFrame API that is similar in spirit to pandas, but designed for distributed execution.

By the end of this practical, you should be able to:

1. start a local Spark session for structured data work;
2. load public CSV and gzip data files in online and local notebook environments;
3. create Spark DataFrames from pandas data and from RDD rows;
4. use DataFrame selection, filtering, grouping, and aggregation;
5. create temporary views and query them with Spark SQL;
6. compare SQL queries with equivalent DataFrame operations.


<a id="2-setup-and-data-files"></a>

### 2. Setup and Data Files

This notebook uses PySpark, pandas, and two public SIT742 data files: `mtcars.csv` and `kddcup.gz`. Choose the execution option that matches where you are running the notebook.

#### Option A: Google Colab / online execution

Use this option when you are running the notebook in Google Colab or another online notebook environment, or when you do not have the SIT742 repository cloned locally. The setup code downloads the required files from the public SIT742 GitHub repository into the notebook runtime and installs PySpark if required. Spark also needs a Java runtime; the setup code checks for Java and uses the online runtime's package manager when available.

#### Option B: Local repository execution

Use this option only if you have cloned the SIT742 repository locally and are running the notebook from its original folder structure. The file paths below are relative to this notebook location. Local execution expects Java, pandas, and PySpark to be available in your active Python environment.

Keep `EXECUTION_MODE = "online"` for Option A. Change it to `"local"` only for Option B.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import shutil
import subprocess
import sys
import tempfile

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.
PYSPARK_PACKAGE = "pyspark==3.5.1"
required_files = ["mtcars.csv", "kddcup.gz"]


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m04g_data_"))
    downloaded_paths = {}
    for filename in required_files:
        url = f"{PUBLIC_DATA_BASE_URL}/{filename}"
        local_file = data_dir / filename
        urlretrieve(url, local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([base / "Jupyter" / "data", base / "data", base / "SIT742" / "Jupyter" / "data"])

    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate, {filename: candidate / filename for filename in required_files}

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the public SIT742 data files locally. "
        "Set EXECUTION_MODE = 'online' or run this notebook from a cloned SIT742 repository.\n"
        f"Checked:\n{checked}"
    )


def ensure_java_available():
    if shutil.which("java"):
        return
    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "openjdk-17-jdk-headless"], check=True)
        return
    raise RuntimeError(
        "Java is required for Spark. Use Google Colab, or install a supported Java runtime locally."
    )


def ensure_python_module(module_name, package_name):
    if importlib.util.find_spec(module_name) is not None:
        return
    if EXECUTION_MODE == "online":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return
    raise RuntimeError(
        f"{package_name} is required. Install it in the active environment with: python -m pip install {package_name}"
    )


if EXECUTION_MODE == "online":
    DATA_DIR, data_files = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR, data_files = find_local_data_dir(required_files)
else:
    raise ValueError("EXECUTION_MODE must be either 'online' or 'local'.")

ensure_java_available()
ensure_python_module("pandas", "pandas")
ensure_python_module("pyspark", PYSPARK_PACKAGE)

mtcars_path = data_files["mtcars.csv"]
kdd_path = data_files["kddcup.gz"]
DataSet = mtcars_path
KDD_DATASET = kdd_path

print("Execution mode:", EXECUTION_MODE)
print("Data directory:", DATA_DIR)
print("mtcars file:", mtcars_path, mtcars_path.stat().st_size, "bytes")
print("KDD file:", kdd_path, kdd_path.stat().st_size, "bytes")


<a id="3-spark-session-and-dataframe-setup"></a>

### 3. Spark Session and DataFrame Setup

A Spark application starts through a `SparkSession`. The session gives access to a `SparkContext` for RDD work and to Spark SQL functionality for DataFrame and SQL queries.

This notebook uses `SparkSession` as the main entry point. A `SQLContext` object is also created so that older Spark SQL terminology is visible, but new examples should prefer `spark` and `spark.sql(...)`.


In [ ]:
from pyspark.sql import SparkSession, SQLContext, Row
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SIT742-M04G-SparkSQL")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
sqlContext = SQLContext(sc)

print("Spark version:", spark.version)
print("Spark master:", sc.master)
print("SparkContext type:", type(sc))
print("SQLContext type:", type(sqlContext))


#### 3.1 Load `mtcars` with pandas

The `mtcars` data file contains 32 cars and the variables used in many introductory data-analysis examples. The file includes a `car` name column plus numeric columns such as miles per gallon, cylinder count, horsepower, weight, and transmission type.


In [ ]:
import pandas as pd

mtcars = pd.read_csv(mtcars_path)
print("mtcars shape:", mtcars.shape)
mtcars.head()


#### 3.2 Create Spark DataFrames

Spark can create a DataFrame from local Python or pandas data. It can also read the CSV file directly. The examples below keep the original teaching idea of comparing both approaches.


In [ ]:
# Create a Spark DataFrame from the local pandas DataFrame.
sdf = spark.createDataFrame(mtcars)
sdf.printSchema()

# Read the same CSV file directly with Spark.
sdf2 = spark.read.option("header", True).option("inferSchema", True).csv(str(mtcars_path))
sdf3 = spark.read.format("csv").option("header", True).option("inferSchema", True).load(str(mtcars_path))

print("sdf2 rows:", sdf2.count())
print("sdf3 rows:", sdf3.count())


<a id="4-dataframe-operations-on-mtcars"></a>

### 4. DataFrame Operations on `mtcars`

Spark DataFrames support familiar structured-data operations such as selecting columns, filtering rows, creating calculated columns, grouping, and aggregation.


In [ ]:
# Display rows and select one column.
sdf.show(5)
sdf.select("mpg").show(5)

# Filter cars with miles per gallon below 18.
sdf.filter(F.col("mpg") < 18).show(5)


In [ ]:
# Create a new column with weight converted from thousands of pounds to approximate metric tonnes.
sdf.withColumn("wtTon", F.col("wt") * 0.45).show(6)


In [ ]:
# Compute average weight by cylinder count.
sdf.groupBy("cyl").agg(F.avg("wt").alias("avg_wt")).orderBy("cyl").show()

# Count cars by cylinder count and sort by the most common values.
sdf.groupBy("cyl").count().orderBy(F.desc("count")).show()


<a id="5-sql-queries-with-temporary-views"></a>

### 5. SQL Queries with Temporary Views

A Spark DataFrame can be registered as a temporary view. After that, SQL statements can query the view and return Spark DataFrames.


In [ ]:
# Register this DataFrame as a temporary SQL view.
sdf.createOrReplaceTempView("cars")

highgearcars = spark.sql("""
    SELECT car, cyl, gear
    FROM cars
    WHERE cyl >= 4 AND cyl <= 9
    ORDER BY gear DESC, car
""")
highgearcars.show(8, truncate=False)


<a id="6-kdd-sparksql-application"></a>

### 6. KDD SparkSQL Application

This section uses the reduced KDD Cup 1999 network-interaction dataset. The public SIT742 copy is provided as `kddcup.gz` so that the notebook can run without requiring a separate manual upload.

The original KDD Cup 1999 information page is available from the KDD archive: http://kdd.ics.uci.edu/databases/kddcup99/kddcup99.html


#### 6.1 Getting the data and creating the RDD

Spark can read a gzip text file directly. Each line is a comma-separated network-interaction record.


In [ ]:
data_file = str(kdd_path)
raw_data = sc.textFile(data_file).cache()
kdd_line_count = raw_data.count()

print("KDD line count:", kdd_line_count)
print("First record:", raw_data.first()[:120] + "...")


#### 6.2 Creating a DataFrame from parsed rows

Spark SQL can convert an RDD of `Row` objects to a DataFrame. Here we select a small set of columns from the KDD records so that the example stays focused on Spark SQL operations.


In [ ]:
csv_data = raw_data.map(lambda line: line.split(","))
parsed_data = csv_data.filter(lambda fields: len(fields) > 41)

row_data = parsed_data.map(lambda fields: Row(
    duration=int(fields[0]),
    protocol_type=fields[1],
    service=fields[2],
    flag=fields[3],
    src_bytes=int(fields[4]),
    dst_bytes=int(fields[5])
))

interactions_df = spark.createDataFrame(row_data).cache()
interactions_count = interactions_df.count()
interactions_df.createOrReplaceTempView("interactions")

print("interactions rows:", interactions_count)
interactions_df.printSchema()


#### 6.3 Querying the KDD DataFrame with SQL

Now we can run SQL queries over the temporary view. The example below selects TCP interactions with long duration and no destination-byte transfer.


In [ ]:
tcp_interactions = spark.sql("""
    SELECT duration, dst_bytes
    FROM interactions
    WHERE protocol_type = 'tcp' AND duration > 1000 AND dst_bytes = 0
""")
tcp_interactions.show(10)


In [ ]:
# Query results are Spark DataFrames. Their underlying RDDs can still be used when needed.
tcp_interactions_out = tcp_interactions.rdd.map(
    lambda row: "Duration: {}, Dest. bytes: {}".format(row.duration, row.dst_bytes)
)
for row_text in tcp_interactions_out.take(10):
    print(row_text)


#### 6.4 Queries as DataFrame operations

Spark DataFrames also provide a method-based query interface. The examples below group and filter the KDD interactions without writing SQL strings.


In [ ]:
from time import time

t0 = time()
interactions_df.select("protocol_type", "duration", "dst_bytes").groupBy("protocol_type").count().show()
print("Query performed in {} seconds".format(round(time() - t0, 3)))


In [ ]:
t0 = time()
(
    interactions_df
    .select("protocol_type", "duration", "dst_bytes")
    .filter(F.col("duration") > 1000)
    .filter(F.col("dst_bytes") == 0)
    .groupBy("protocol_type")
    .count()
    .show()
)
print("Query performed in {} seconds".format(round(time() - t0, 3)))


#### 6.5 Adding a label column for exploratory grouping

The final field in each KDD record is a detailed interaction label. The helper function below maps those labels into two broad groups: `normal` and `attack`.


In [ ]:
def get_label_type(label):
    if label != "normal.":
        return "attack"
    return "normal"


row_labeled_data = parsed_data.map(lambda fields: Row(
    duration=int(fields[0]),
    protocol_type=fields[1],
    service=fields[2],
    flag=fields[3],
    src_bytes=int(fields[4]),
    dst_bytes=int(fields[5]),
    label=get_label_type(fields[41])
))

interactions_labeled_df = spark.createDataFrame(row_labeled_data).cache()
interactions_labeled_df.printSchema()


In [ ]:
t0 = time()
label_counts_df = interactions_labeled_df.groupBy("label").count().orderBy("label")
label_counts_df.show()
print("Query performed in {} seconds".format(round(time() - t0, 3)))


In [ ]:
t0 = time()
(
    interactions_labeled_df
    .groupBy("label", "protocol_type")
    .count()
    .orderBy("label", "protocol_type")
    .show()
)
print("Query performed in {} seconds".format(round(time() - t0, 3)))


In [ ]:
t0 = time()
(
    interactions_labeled_df
    .withColumn("dst_bytes_is_zero", F.col("dst_bytes") == 0)
    .groupBy("label", "protocol_type", "dst_bytes_is_zero")
    .count()
    .orderBy("label", "protocol_type", "dst_bytes_is_zero")
    .show()
)
print("Query performed in {} seconds".format(round(time() - t0, 3)))


<a id="7-checks-and-reflection"></a>

### 7. Checks and Reflection

Run the cell below after completing the notebook to confirm that the key DataFrames and Spark SQL checks are available.


In [ ]:
label_counts = {row["label"]: row["count"] for row in label_counts_df.collect()}

final_checks = {
    "Spark session is available": spark is not None,
    "mtcars has 32 rows": len(mtcars) == 32,
    "Spark mtcars DataFrame has 32 rows": sdf.count() == 32,
    "KDD file has expected public-data row count": kdd_line_count == 494021,
    "KDD interactions DataFrame matches raw row count": interactions_count == kdd_line_count,
    "KDD labels include normal and attack": {"normal", "attack"}.issubset(label_counts.keys()),
}

for check_name, passed in final_checks.items():
    print(f"{check_name}: {'OK' if passed else 'check again'}")

assert all(final_checks.values())


<a id="8-reflection-and-references"></a>

### 8. Reflection and References

Before moving on, consider these questions:

1. Which parts of the notebook use SQL, and which parts use DataFrame operations?
2. Why is a temporary view useful when working with Spark SQL?
3. What changes when the input data grows from a small CSV file to a larger gzip text file?
4. Why should `take()` or grouped summaries be preferred over collecting a large DataFrame or RDD to the driver?

References:

- Apache Spark SQL documentation: https://spark.apache.org/docs/latest/sql-programming-guide.html
- PySpark DataFrame API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html
- KDD Cup 1999 data archive: http://kdd.ics.uci.edu/databases/kddcup99/kddcup99.html
